## LSTM  Long Short-Term Memory

The aim of this notebook is to show how to train LSTM on a real synthetic dataset.

### The Training Problem

The problem consist in training a LSTM model to classify a sequence of terms, the two possible outputs are: valid and invalid.

BNF Definition below:

$$
\begin{array}{rcl}
\langle\mathit{string}\rangle   & \mathrel{::=} & \langle\mathit{term}\rangle \\
                              & \mid          & \langle\mathit{string}\rangle \mathbin{\texttt{+}} \langle\mathit{term}\rangle \\[2pt]
\langle\mathit{term}\rangle   & \mathrel{::=} & AB, ED, CK \\[2pt]
\end{array}
$$

The goal is to see if at inference time the model respect the grammar rules and generate valid sequences.

In [1]:
"""Data Generation Utils"""

import numpy as np

ALPHABET = ["A", "B", "D", "E", "K"]
TERMS = ["AB", "ED", "CK"]

def generate_valid_seq(length: int) -> str:
    out = ""
    for _ in range(length):
        ri, = np.random.randint(0, len(TERMS), 1, dtype=int)
        out = f"{out}{TERMS[ri]}"

    return out

def generate_valid_seqs(row: int, col: int) -> list[str]:
    outs = []
    for _ in range(row):
        out = ""
        for _ in range(col):
            ri, = np.random.randint(0, len(TERMS), 1, dtype=int)
            out = f"{out}{TERMS[ri]}"
        outs.append(out)
    
    return outs

def generate_invalid_seq(length: int) -> str:
    out = ""
    for _ in range(length):
        ri, = np.random.randint(0, len(ALPHABET), 1, dtype=int)
        out = f"{out}{ALPHABET[ri]}"

    return out

def generate_invalid_seqs(row: int, col: int) -> list[str]:
    outs = []
    for _ in range(row):
        out = ""
        for _ in range(col):
            ri, = np.random.randint(0, len(ALPHABET), 1, dtype=int)
            out = f"{out}{ALPHABET[ri]}"
        outs.append(out)
    
    return outs

In [2]:
"""Grammar Parser"""

def parse(input: str) -> bool:
    prev = None

    for c in input:
        match c:
            case 'A':
                prev = "A"
            case 'B':
                if prev != "A": return False
                prev = ""
            case 'C':
                prev = "C"
            case 'D':
                if prev != "E": return False
                prev = ""
            case 'E':
                prev = "E"
            case 'K':
                if prev != "C": return False
                prev = ""
            
    return True

## Tokenizer

In order to represent the terms of the language we need to define a tokenizer that will map every letters of the alphabet to point in a vector space.
Our vector space has 4 dimensions:

- Is Vowel
- Is Consonant
- Is Alone (if the terms were letter appear have length 1)
- Position in alphabet

so every term will be represented as a 4D vector in this space.

In [3]:
"""Tokenizer"""

VOWELS = {"A", "E"}

def tokenize(input: str | list[str] | list[list[str]], _alone: bool = True) -> np.ndarray:
    # list of strings / list of lists of strings: recurse elementwise and stack
    if isinstance(input, list):
        return np.stack([tokenize(elem, _alone=True) for elem in input])

    # multi-character string: recurse letter by letter (none of them stand alone)
    if isinstance(input, str) and len(input) > 1:
        return np.stack([tokenize(letter, _alone=False) for letter in input])

    # base case: a single character
    is_vowel = input in VOWELS
    is_consonant = input in ALPHABET and not is_vowel
    is_alone = _alone
    position = ALPHABET.index(input) + 1 if input in ALPHABET else 0

    return np.array([is_vowel, is_consonant, is_alone, position], dtype=np.float32)


In [4]:
"""Dataset Generation"""

from thorcino.dataset.dataset import DataLoader, TensorDataset
from thorcino.tensor import Tensor


N_SEQ = 50
SPLIT_RATIO = 0.9
BATCH_SIZE = 16
TRAIN_LEN = int(N_SEQ*SPLIT_RATIO)
ROW, COL = 25, 25

X_valid = np.array(tokenize(generate_valid_seqs(ROW, COL)))
X_invalid = np.array(tokenize(generate_invalid_seqs(ROW, COL*2)))

Y_valid = np.ones((X_valid.shape[0], 1))
Y_invalid = np.zeros((X_invalid.shape[0], 1))

X = np.append(X_valid, X_invalid, axis=0)
Y = np.append(Y_valid, Y_invalid, axis=0)

print(f"X shape = {X.shape}")
print(f"Y shape = {Y.shape}")

X_train, X_test = X[:TRAIN_LEN], X[TRAIN_LEN:]
Y_train, Y_test = Y[:TRAIN_LEN], Y[TRAIN_LEN:]

print(X_train.shape, Y_train.shape)

train_dataset, test_dataset = TensorDataset(Tensor(X_train), Tensor(Y_train)), TensorDataset(Tensor(X_test), Tensor(Y_test))
train_dataloader, test_dataloader = DataLoader(train_dataset, BATCH_SIZE, True), DataLoader(test_dataset, BATCH_SIZE, True)

X shape = (50, 50, 4)
Y shape = (50, 1)
(45, 50, 4) (45, 1)


In [5]:
"""Model Architecture"""

from thorcino.activations import Sigmoid
from thorcino.layers.linear import Linear
from thorcino.layers.lstm import LSTM
from thorcino.layers.sequential import Sequential
from thorcino.losses import MSELoss
from thorcino.optimizer import SGD
from thorcino.training.schedulers import CosineSchedule
from thorcino.training.trainer import Trainer

EPOCHS, EVAL_STEP = 2, 10
MAX_LR, MIN_LR = 1e-3, 1e-5

model = Sequential(
    LSTM(
        in_feature=4,
        hidden_units=4,
        out_type='n_to_1',
    ),
    Linear(
        in_feature=4,
        out_feature=1,
    ),
    Sigmoid()
)
loss = MSELoss()
optimizer = SGD(model.parameters, MAX_LR)
scheduler = CosineSchedule(MIN_LR, MAX_LR, EPOCHS)
trainer = Trainer(
    model,
    loss,
    optimizer,
    scheduler,
)

In [ ]:
"""Training"""

for e in range(EPOCHS):
    trainer.train_epoch(train_dataloader)

    if e%EVAL_STEP == 0:
        trainer.eval(test_dataloader)